# Gravity and Magnetics over a Tunnel

**Near-Surface Geophysics — Missouri S&T**

A tunnel runs through bedrock under a field. Can gravity and magnetic
measurements made on the surface find it — and if they can, what can they
tell us about it?

This notebook works the problem in both directions. First the **forward
problem**: build a model of the ground with the tunnel in it, and compute
the gravity and magnetic anomalies it would produce along a profile across
it. Then add realistic noise to those numbers, forget the model, and do the
**inverse problem**: recover a density model and a magnetic susceptibility
model from the noisy data alone, and compare them with the truth.

The model:

* **1 m of soil** over **bedrock**.
* A **tunnel** in the bedrock whose roof is **3 m below the top of the
  rock**, so **4 m below the ground surface**.
* The tunnel is **3 m high and 4 m wide**, and it is **long**: it runs
  straight on, at right angles to our profile, for much further than it is
  deep. That makes the problem **2-D** — every cross-section along the
  tunnel looks the same, and one profile across it tells the whole story.

Sections:

1. **The model** — rock properties, and why only the tunnel shows up.
2. **The forward problem** — gravity and magnetic anomalies of the tunnel,
   and what each station is sensitive to.
3. **Noisy observations** — what a survey would actually measure.
4. **The inverse problem** — many models fit the data; regularisation
   chooses one. Depth weighting, and choosing the trade-off parameter β.
5. **Adding what we know** — bounds and compactness sharpen the picture.
6. **The density and susceptibility models**, and what they do and do not
   tell us.
7. **Things to try.**

Units are **metres**, densities in **g/cm³**, gravity in **mGal**
(1 mGal = 1000 µGal = 10⁻⁵ m/s²), magnetic fields in **nT**, and
susceptibility is dimensionless **SI**. Depth is positive **down**.

> **The one thing to take away.** Potential-field data do not determine a
> model. Infinitely many models fit them equally well, and the inversion
> returns the one that best matches the *rules you gave it*. Change the
> rules and the picture changes. What survives every reasonable set of
> rules — here, where the tunnel is and how much mass is missing — is what
> the data actually know.

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or out of
date) and imports what the notebook uses. Everything here needs only
NumPy, SciPy and Matplotlib; the whole notebook runs in under a minute.

In [ ]:
# Checks for the *potentialfields* code specifically: an older install imports
# fine and then lacks it, and since the version string does not change
# between builds, a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.potentialfields import LinearInversion  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.potentialfields is still missing after installing. "
        "In Colab: Runtime > Restart session, then run this cell again."
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from matplotlib.patches import Rectangle

from shallowgeo.potentialfields import (
    GRAVITATIONAL_CONSTANT, DEPTH_EXPONENT, LinearInversion, Mesh2D,
    depth_weights, field_direction, gravity_kernel, magnetic_kernel, make_mesh,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False,
                     "axes.spines.top": False, "axes.spines.right": False})
# Observed data in ink; model curves in three hues that stay distinct for
# colour-blind readers.
INK, BLUE, ORANGE, AQUA = "0.15", "#2a78d6", "#eb6834", "#1baf7a"
print("ready")

## 1. The model

Every number that defines the problem is in the next cell. Change any of
them and re-run the notebook from here down — section 7 suggests some
experiments.

**Rock properties.** The bedrock here is a crystalline rock such as the
granite and rhyolite of the St. Francois Mountains, which is weakly
magnetic. The tunnel is air: no density, no susceptibility. The soil is
lighter than the rock and a little magnetic, as soils usually are.

| Unit | Density (g/cm³) | Susceptibility (SI) |
|---|---|---|
| Soil | 1.80 | 0.0005 |
| Bedrock | 2.65 | 0.005 |
| Tunnel (air) | 0 | 0 |

**The survey.** One profile, 60 m long, centred on the tunnel and run
south to north, so the tunnel runs east–west. Gravity is read every 2 m
(a CG-5 reading takes a few minutes, so gravity stations are always
sparser), with the sensor 0.2 m above the ground. Magnetics is read every
1 m with the sensor on a staff 1 m up.

**The Earth's field** in central Missouri is about 52 000 nT, dipping 64°
below horizontal toward the north, with a declination close to zero. (For
your own site, use the NOAA magnetic-field calculator.)

In [ ]:
# --- Geometry (metres) ---------------------------------------------------
SOIL_THICKNESS = 1.0           # soil over bedrock
TUNNEL_ROOF_BELOW_ROCK = 3.0   # bedrock above the tunnel roof
TUNNEL_HEIGHT = 3.0
TUNNEL_WIDTH = 4.0
TUNNEL_X = 0.0                 # centre of the tunnel along the profile

# --- Properties: (density g/cm^3, susceptibility SI) ---------------------
SOIL = (1.80, 0.0005)
ROCK = (2.65, 0.005)
AIR = (0.0, 0.0)

# --- Survey ---------------------------------------------------------------
GRAV_X = np.arange(-30, 30.01, 2.0)     # gravity stations
GRAV_HEIGHT = 0.2                       # gravimeter sensor above ground
MAG_X = np.arange(-30, 30.01, 1.0)      # magnetometer stations
MAG_HEIGHT = 1.0                        # magnetometer sensor above ground

# --- Earth's field ------------------------------------------------------
FIELD = 52_000.0         # nT
INCLINATION = 64.0       # degrees below horizontal
DECLINATION = 0.0        # degrees east of north
PROFILE_AZIMUTH = 0.0    # direction of increasing x: 0 = south to north

tunnel_top = SOIL_THICKNESS + TUNNEL_ROOF_BELOW_ROCK
tunnel_bottom = tunnel_top + TUNNEL_HEIGHT
tunnel_left = TUNNEL_X - TUNNEL_WIDTH / 2
tunnel_right = TUNNEL_X + TUNNEL_WIDTH / 2
print(f"tunnel from {tunnel_left:g} to {tunnel_right:g} m along the profile, "
      f"{tunnel_top:g} to {tunnel_bottom:g} m deep")

### The mesh

To compute anything — and later to invert — the ground is divided into
cells. Each cell is a long prism running along the tunnel, with a square
cross-section 0.5 m on a side, and holds one value of density or
susceptibility. The core of the mesh runs 5 m past each end of the
profile and 15 m deep; a few cells that grow in size are added around the
outside (*padding*) so that the edge of the mesh is not mistaken for
geology.

In [ ]:
mesh = make_mesh(-35, 35, 15, cell=0.5)
nz, nx = mesh.shape
print(f"{mesh.n_cells} cells: {nx} across by {nz} down; "
      f"mesh spans x = {mesh.x_edges[0]:.0f} to {mesh.x_edges[-1]:.0f} m, "
      f"z = 0 to {mesh.z_edges[-1]:.0f} m")

soil = mesh.layer(0, SOIL_THICKNESS)           # fraction of each cell in each unit
tunnel = mesh.rectangle(tunnel_left, tunnel_right, tunnel_top, tunnel_bottom)
rock = 1 - soil - tunnel

density = soil * SOIL[0] + rock * ROCK[0] + tunnel * AIR[0]
susceptibility = soil * SOIL[1] + rock * ROCK[1] + tunnel * AIR[1]

In [ ]:
def section(ax, values, *, cmap="RdBu_r", vmin=None, vmax=None, norm=None, title="",
            label="", xlim=(-20, 20), zmax=12, outline=True):
    """Draw a model on the mesh, with the true tunnel and rock surface outlined."""
    if norm is None and vmin is None and cmap == "RdBu_r":     # diverging: centre on zero
        vmax = np.abs(values).max() if vmax is None else vmax
        vmin = -vmax
    if norm is not None:
        vmin = vmax = None
    pc = ax.pcolormesh(mesh.x_edges, mesh.z_edges, values.reshape(mesh.shape),
                       cmap=cmap, vmin=vmin, vmax=vmax, norm=norm, shading="flat")
    if outline:
        ax.add_patch(Rectangle((tunnel_left, tunnel_top), TUNNEL_WIDTH, TUNNEL_HEIGHT,
                               fill=False, ec="k", lw=1))
        ax.axhline(SOIL_THICKNESS, color="k", lw=0.6, ls=":")
    ax.set(xlim=xlim, ylim=(zmax, 0), aspect="equal", title=title,
           xlabel="distance along profile (m, south → north)", ylabel="depth (m)")
    plt.colorbar(pc, ax=ax, label=label, shrink=0.8, pad=0.02)
    return pc


fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
section(axs[0], density, cmap="Blues", vmin=0, vmax=2.8,
        title="Density", label="g/cm³")
section(axs[1], susceptibility, cmap="Purples", vmin=0, vmax=0.0055,
        title="Magnetic susceptibility", label="SI")
fig.tight_layout()

### Only the tunnel makes an anomaly

The soil and the rock are both there, but on a flat site **flat layers
produce no anomaly**:

* **Gravity.** An infinite flat layer pulls down with the same
  $2\pi G \rho t$ at every station — the Bouguer slab. That is a constant
  added to every reading. Gravity surveys measure only *differences* from a
  base station, so the constant disappears.
* **Magnetics.** An infinite flat layer magnetised uniformly makes **no
  external field at all**: the magnetic poles on its top and bottom surfaces
  cancel exactly.

So what the survey sees is the *difference* between the real ground and
the layered ground without the tunnel — the **contrast** of the tunnel
against the rock around it:

$$\Delta\rho = 0 - 2.65 = -2.65\ \text{g/cm}^3, \qquad
  \Delta\chi = 0 - 0.005 = -0.005\ \text{SI}.$$

The cell below checks both claims on a very wide two-layer model, then
builds the contrast models we will use from here on. A **void is a
negative contrast**: missing mass, missing magnetisation.

In [ ]:
wide = Mesh2D([-1e7, 1e7], [0, SOIL_THICKNESS, 50.0])   # soil over 49 m of rock, 20 000 km wide
layered_rho = np.array([SOIL[0], ROCK[0]])
layered_chi = np.array([SOIL[1], ROCK[1]])
g_layers = gravity_kernel(wide, GRAV_X, GRAV_HEIGHT) @ layered_rho
t_layers = magnetic_kernel(wide, MAG_X, MAG_HEIGHT, field=FIELD,
                           inclination=INCLINATION, declination=DECLINATION,
                           profile_azimuth=PROFILE_AZIMUTH) @ layered_chi
print(f"layers alone, gravity:   {g_layers.min():.4f} to {g_layers.max():.4f} mGal "
      f"(spread {np.ptp(g_layers) * 1e3:.2g} µGal)")
print(f"layers alone, magnetics: largest |anomaly| {np.abs(t_layers).max():.1e} nT")

background_rho = soil * SOIL[0] + (1 - soil) * ROCK[0]
background_chi = soil * SOIL[1] + (1 - soil) * ROCK[1]
true_rho = density - background_rho          # contrast: -2.65 in the tunnel, 0 elsewhere
true_chi = susceptibility - background_chi   # contrast: -0.005 in the tunnel
print(f"\ndensity contrast of the tunnel:        {true_rho.min():.2f} g/cm³")
print(f"susceptibility contrast of the tunnel: {true_chi.min():.4f} SI")

## 2. The forward problem

Given a model, compute the data. Both gravity and magnetics are **linear**
in the physical property: double every density and every gravity anomaly
doubles. So for a mesh, each datum is a weighted sum of the cell values,

$$d_i = \sum_j G_{ij}\, m_j \qquad\text{or}\qquad \mathbf d = \mathbf G\,\mathbf m,$$

where $G_{ij}$ is the anomaly at station $i$ from cell $j$ with unit
property. Build $\mathbf G$ once and every forward model is one matrix
multiplication. Row $i$ of $\mathbf G$ says how much station $i$ "sees" each
cell; we will look at that below.

### Gravity

A long horizontal line of mass $\lambda$ per metre, at depth $z$ below a
station and horizontal offset $x$, pulls down with

$$g_z = 2G\lambda\,\frac{z}{x^2 + z^2}.$$

Integrating that over a rectangular cell of density $\rho$ gives a closed
form (Telford et al., 1990; Blakely, 1995) — a sum over the four corners of
$x\ln r + z\arctan(x/z)$, with $x$, $z$ measured from the station to each
corner. That is what `gravity_kernel` evaluates, in mGal per g/cm³.

In [ ]:
G_grav = gravity_kernel(mesh, GRAV_X, GRAV_HEIGHT)
grav_true = G_grav @ true_rho
print(f"G is {G_grav.shape[0]} stations x {G_grav.shape[1]} cells")
print(f"peak anomaly {grav_true.min():.4f} mGal = {grav_true.min() * 1e3:.0f} µGal")

How does that compare with the simplest possible model, a **line of
missing mass** at the centre of the tunnel with the same deficit per metre
($\Delta\lambda = \Delta\rho \times$ width $\times$ height)? Below the two
curves are plotted together, and they are nearly the same — the
difference is a few µGal at most, below what a gravimeter can measure.

This is the first hint of the difficulty ahead: **from the surface, a
4 × 3 m box looks like a line**. The gravity profile knows roughly where
the missing mass is and how much is missing; it knows almost nothing about
its shape.

There is also a rule of thumb hidden in that formula: for a line mass the
anomaly falls to half its peak at a horizontal distance equal to the
depth. Measuring the **half-width** of the anomaly is a quick estimate of
how deep the centre of the body is.

In [ ]:
x_fine = np.linspace(-30, 30, 601)
dlam = ROCK[0] * 1e3 * TUNNEL_WIDTH * TUNNEL_HEIGHT        # kg per metre of tunnel
zc = (tunnel_top + tunnel_bottom) / 2 + GRAV_HEIGHT        # centre below the sensor
line_mass = -2 * GRAVITATIONAL_CONSTANT * dlam * zc / ((x_fine - TUNNEL_X)**2 + zc**2) * 1e5
box = gravity_kernel(mesh, x_fine, GRAV_HEIGHT) @ true_rho

half = x_fine[box <= box.min() / 2]
half_width = (half.max() - half.min()) / 2
print(f"largest box - line difference: {np.abs(box - line_mass).max() * 1e3:.1f} µGal")
print(f"half-width {half_width:.1f} m  ->  centre about {half_width - GRAV_HEIGHT:.1f} m deep "
      f"(true {zc - GRAV_HEIGHT:.1f} m)")

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(x_fine, box * 1e3, color=BLUE, lw=2, label="4 × 3 m tunnel")
ax.plot(x_fine, line_mass * 1e3, color=ORANGE, lw=1.5, ls="--", label="line of missing mass")
ax.axhline(box.min() / 2 * 1e3, color="0.6", lw=0.8)
ax.annotate("half the peak", (half.max(), box.min() / 2 * 1e3), xytext=(8, -14),
            textcoords="offset points", color="0.35", fontsize=9)
ax.set(xlabel="distance along profile (m)", ylabel="gravity anomaly (µGal)",
       title="Gravity anomaly of the tunnel")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()

### Magnetics

Rock in the Earth's field $\mathbf F$ becomes magnetised along that field,
with magnetisation $\mathbf M = \chi\,\mathbf F/\mu_0$ (*induced*
magnetisation; we assume no remanence, and $\chi \ll 1$ so the rock does not
noticeably distort the field inside itself). A body with a susceptibility
contrast becomes a magnet, and its field adds to the Earth's.

A total-field magnetometer such as the G-857 measures the **strength** of
the total field, so what it records as the anomaly is the component of the
body's field **along** $\mathbf F$:

$$\Delta T \approx \mathbf B_{\text{body}} \cdot \hat{\mathbf F}.$$

Three things make the magnetic anomaly look different from the gravity one:

* **It is a dipole field.** The body has a north and a south pole, so the
  anomaly has a high and a low.
* **It depends on direction.** Our profile runs south to north, and the
  field dips 64° to the north. A *magnetic* body would give a high on the
  south side and a low on the north; the tunnel is a *missing* magnet, so
  the signs flip: **a low to the south, a high to the north**, crossing
  zero roughly over the tunnel.
* **It decays faster.** A 2-D gravity source falls off as $1/r$; a 2-D
  magnetic dipole as $1/r^2$. Magnetics is more sensitive to shallow
  sources and less to deep ones.

In 2-D only the part of the field in the plane of the profile matters:
magnetisation *along* the tunnel makes no external field, just as for the
flat layer. So the anomaly depends on which way the tunnel runs. Ours runs
east–west, so a south–north profile across it sees both the vertical and
the northward parts of the field. If the tunnel ran north–south instead
(crossed by a west–east profile), the northward part of the field would lie
along the tunnel and drop out. Only the vertical part would count, and the
anomaly would be a **symmetric** low with small positive shoulders. Here the
field is steep (the vertical part is 90 % of it), so the two anomalies are
about the same size and differ mainly in shape. Near the magnetic equator
the difference would be dramatic. The plot shows both.

Either way the profile has to **cross** the tunnel. A profile run *along* a
long tunnel reads the same value at every station, a constant that looks
just like a change of base level, and finds nothing.

In [ ]:
mag_args = dict(field=FIELD, inclination=INCLINATION, declination=DECLINATION)
G_mag = magnetic_kernel(mesh, MAG_X, MAG_HEIGHT, profile_azimuth=PROFILE_AZIMUTH, **mag_args)
mag_true = G_mag @ true_chi

fx, fy, fz = field_direction(INCLINATION, DECLINATION, PROFILE_AZIMUTH)
print(f"field direction (along profile, along tunnel, down) = ({fx:.2f}, {fy:.2f}, {fz:.2f})")
print(f"anomaly from {mag_true.min():.1f} to {mag_true.max():.1f} nT")

ew = magnetic_kernel(mesh, x_fine, MAG_HEIGHT, profile_azimuth=PROFILE_AZIMUTH + 90, **mag_args)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(x_fine, magnetic_kernel(mesh, x_fine, MAG_HEIGHT, profile_azimuth=PROFILE_AZIMUTH,
                                **mag_args) @ true_chi,
        color=BLUE, lw=2, label="our profile (across an east–west tunnel)")
ax.plot(x_fine, ew @ true_chi, color=ORANGE, lw=1.5, ls="--",
        label="tunnel running north–south (profile west → east)")
ax.axhline(0, color="0.6", lw=0.8)
ax.axvspan(tunnel_left, tunnel_right, color="0.9", zorder=0)
ax.set(xlabel="distance along profile (m)", ylabel="total-field anomaly (nT)",
       title="Magnetic anomaly of the tunnel (grey: above the tunnel)")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()

### What one station sees

Each row of $\mathbf G$ is a map of how much one station responds to each
cell. Here is the row for the station directly above the tunnel, for each
method, divided by cell area so the large padding cells do not stand out.
The colour scale is **logarithmic** — each shade is ten times the next —
because otherwise only the few cells right under the sensor would show. On
the right, the same sensitivity straight down from the station, on log
axes: it falls off as $1/z$ for gravity and $1/z^2$ for magnetics.

This picture explains most of what follows. A cell near the surface moves
the reading far more than a deep one, so a deep body can always be traded
for a weaker, shallower one. And the magnetic sensitivity changes sign: a
cell to the south of the station pushes the reading the opposite way from
a cell beneath it.

In [ ]:
i_g = np.argmin(np.abs(GRAV_X - TUNNEL_X))
i_m = np.argmin(np.abs(MAG_X - TUNNEL_X))
sens_g = G_grav[i_g] / mesh.area
sens_m = G_mag[i_m] / mesh.area

fig = plt.figure(figsize=(14, 3.6))
ax1 = fig.add_subplot(1, 3, 1)
section(ax1, sens_g / sens_g.max(), cmap="Blues", norm=LogNorm(1e-2, 1), xlim=(-12, 12),
        title=f"gravity station at x = {GRAV_X[i_g]:g} m", label="relative sensitivity")
ax2 = fig.add_subplot(1, 3, 2)
section(ax2, sens_m / np.abs(sens_m).max(), norm=SymLogNorm(1e-3, vmin=-1, vmax=1), xlim=(-12, 12),
        title=f"magnetic station at x = {MAG_X[i_m]:g} m", label="relative sensitivity")

ax3 = fig.add_subplot(1, 3, 3)
col = np.argmin(np.abs(mesh.x_centers - TUNNEL_X))
z = mesh.z_centers
for s, h, colour, name in [(sens_g, GRAV_HEIGHT, BLUE, "gravity"),
                           (sens_m, MAG_HEIGHT, ORANGE, "magnetics")]:
    profile = np.abs(s.reshape(mesh.shape)[:, col])
    ax3.loglog(z + h, profile / profile[0], color=colour, lw=2, label=name)
r = np.array([1.5, 30])
ax3.loglog(r, 1.5 / r, color="0.6", lw=0.8, ls="--")
ax3.loglog(r, (1.5 / r)**2, color="0.6", lw=0.8, ls=":")
ax3.text(12, 1.5 / 12 * 1.3, "1/r", color="0.4"); ax3.text(12, (1.5 / 12)**2 * 1.3, "1/r²", color="0.4")
ax3.set(xlabel="distance below the sensor (m)", ylabel="sensitivity (relative)",
        title="straight down from the station")
ax3.legend(frameon=False)
fig.tight_layout()

## 3. Noisy observations

Real readings are never exact. We add random, normally distributed noise
with standard deviations typical of a careful survey:

* **Gravity: σ = 0.005 mGal (5 µGal)** — the repeatability of a CG-5 after
  drift and tide corrections.
* **Magnetics: σ = 0.5 nT** — instrument noise plus the small-scale
  variation from magnetic soil and stones near the sensor.

This is still optimistic. It leaves out everything a real survey has to
remove before an inversion: instrument drift and Earth tides, diurnal
variation of the magnetic field, terrain, the regional trend from deeper
geology, and fences, pipes and parked cars. A fixed random seed makes the
noise the same every time the notebook is run; change it to see another
realisation.

In [ ]:
GRAV_STD = 0.005   # mGal
MAG_STD = 0.5      # nT
rng = np.random.default_rng(2026)

grav_obs = grav_true + GRAV_STD * rng.standard_normal(grav_true.size)
mag_obs = mag_true + MAG_STD * rng.standard_normal(mag_true.size)

print(f"gravity:   peak {abs(grav_true).max() * 1e3:.0f} µGal, noise {GRAV_STD * 1e3:.0f} µGal "
      f"-> signal/noise {abs(grav_true).max() / GRAV_STD:.0f}")
print(f"magnetics: peak {abs(mag_true).max():.1f} nT, noise {MAG_STD} nT "
      f"-> signal/noise {abs(mag_true).max() / MAG_STD:.0f}")

In [ ]:
def plot_data(ax, x, obs, std, *, unit, scale=1.0, curves=(), title=""):
    """Observed data with one-sigma error bars, and any model curves over them."""
    ax.errorbar(x, obs * scale, yerr=std * scale, fmt="o", ms=3.5, color=INK,
                ecolor="0.6", elinewidth=0.8, capsize=0, label="observed")
    for xc, yc, colour, name, ls in curves:
        ax.plot(xc, yc * scale, color=colour, lw=2, ls=ls, label=name)
    ax.axvspan(tunnel_left, tunnel_right, color="0.92", zorder=0)
    ax.set(xlabel="distance along profile (m)", ylabel=unit, title=title)
    ax.legend(frameon=False, fontsize=9)


fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
plot_data(axs[0], GRAV_X, grav_obs, GRAV_STD, unit="gravity anomaly (µGal)", scale=1e3,
          curves=[(GRAV_X, grav_true, BLUE, "noise-free", "-")], title="Gravity, σ = 5 µGal")
plot_data(axs[1], MAG_X, mag_obs, MAG_STD, unit="total-field anomaly (nT)",
          curves=[(MAG_X, mag_true, BLUE, "noise-free", "-")], title="Magnetics, σ = 0.5 nT")
fig.tight_layout()

## 4. The inverse problem

Now forget the model. All we have is the noisy data, the station
positions, and $\mathbf G$. Find $\mathbf m$.

The trouble is the count:

In [ ]:
print(f"gravity:   {GRAV_X.size} data, {mesh.n_cells} unknowns")
print(f"magnetics: {MAG_X.size} data, {mesh.n_cells} unknowns")

With a hundred times more unknowns than data, $\mathbf d = \mathbf G\mathbf m$
has **infinitely many solutions**, and on top of that the data are noisy, so
we should not even want an exact fit. Potential fields add a deeper
non-uniqueness of their own: Gauss's theorem guarantees that for any body
there is a thinner, shallower layer that produces exactly the same field
outside it.

So the question becomes: of all the models that fit the data **to within
their noise**, which one do we want? We have to say, and we say it with a
*model objective* $\phi_m$ that measures how much we dislike a model. The
inversion minimises

$$\phi(\mathbf m) = \underbrace{\sum_i \left(\frac{(\mathbf G\mathbf m)_i - d_i}{\sigma_i}\right)^2}_{\phi_d\ \text{(data misfit)}}
\;+\; \beta\, \underbrace{\left[\frac{1}{L^2}\int (w\,m)^2\,dA + \int |\nabla(w\,m)|^2\,dA\right]}_{\phi_m\ \text{(model objective)}}$$

* $\phi_d$ is the misfit, each datum weighted by its uncertainty. If the
  model is right and the noise is as stated, its expected value is the
  number of data, $N$.
* $\phi_m$ asks for the **smallest and smoothest** model: values close to
  zero (no anomaly unless the data demand one) that change gradually. $L$ is
  a length, here 2 m, that balances the two.
* $w(z)$ is a **weight** on each cell — coming up in 4.2.
* $\beta$ trades one against the other. Small β: fit the data at all costs,
  noise included. Large β: a smooth, small model that ignores the data.

Because $\mathbf G$ is linear, this has a closed-form solution for any
$\beta$ — a few milliseconds each. That is what `LinearInversion` does.

### 4.1 A first attempt, and why it fails

No weights ($w = 1$), and β chosen so that $\phi_d = N$: fit each datum to
about one standard deviation, no better. (Choosing β is section 4.3.)

In [ ]:
inv_flat = LinearInversion(mesh, G_grav, grav_obs, GRAV_STD, weighting="none")
flat = inv_flat.solve()       # beta chosen so that phi_d = N
print(f"beta = {flat.beta:.3g}, phi_d / N = {flat.chi2:.2f}")
print(f"the true model, for comparison: phi_d / N = {inv_flat.phi_d(true_rho) / GRAV_X.size:.2f}")

fig, axs = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1.25, 1]})
section(axs[0], flat.model, title="Recovered density contrast, no depth weighting",
        label="Δρ (g/cm³)")
plot_data(axs[1], GRAV_X, grav_obs, GRAV_STD, unit="gravity anomaly (µGal)", scale=1e3,
          curves=[(GRAV_X, flat.predicted, BLUE, "this model", "-")], title="Its fit to the data")
fig.tight_layout()

That model fits the data to within their noise — by the only test the data
can apply, it is as good as the true model — and yet it is wrong. The
missing mass is strongest **right at the surface** and fades with depth,
and it is spread over a region several times wider than the tunnel at a
contrast ten times weaker.

It is not a bug. It is exactly what we asked for. Look back at the
sensitivity plot: a cell near the surface moves the reading far more than
a deep one, so the *smallest* model that explains the anomaly puts its
mass as shallow as it can. The data cannot tell the two models apart; only
our rules can, and our rules preferred this one.

### 4.2 Depth weighting

The fix (Li and Oldenburg, 1996) is to make deep cells cheaper, by just
enough to cancel the decay of the kernel. With

$$w(z) = (z + z_0)^{-\nu}$$

in the model objective, a deep cell pays less for being non-zero. The
square of $w$ should fall off like the kernel itself: in 2-D that means
$\nu = 1/2$ for gravity ($1/r$) and $\nu = 1$ for magnetics ($1/r^2$).
($z_0$ is the sensor height.) In 3-D the kernels decay one power faster and
the usual exponents are 1 and 1.5.

In [ ]:
print("depth-weighting exponents:", DEPTH_EXPONENT)
fig, ax = plt.subplots(figsize=(5, 3.4))
for name, h, colour in [("gravity", GRAV_HEIGHT, BLUE), ("magnetics", MAG_HEIGHT, ORANGE)]:
    w = depth_weights(mesh, DEPTH_EXPONENT[name], h).reshape(mesh.shape)[:, 0]
    ax.plot(w, mesh.z_centers, color=colour, lw=2, label=f"{name}, ν = {DEPTH_EXPONENT[name]}")
ax.set(ylim=(15, 0), xlabel="weight w(z)", ylabel="depth (m)", title="Depth weights")
ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
inv_grav = LinearInversion(mesh, G_grav, grav_obs, GRAV_STD, weighting="gravity", z0=GRAV_HEIGHT)
smooth_grav = inv_grav.solve()
print(f"beta = {smooth_grav.beta:.3g}, phi_d / N = {smooth_grav.chi2:.2f}")

fig, axs = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1.25, 1]})
section(axs[0], smooth_grav.model, title="Recovered density contrast, depth-weighted",
        label="Δρ (g/cm³)")
plot_data(axs[1], GRAV_X, grav_obs, GRAV_STD, unit="gravity anomaly (µGal)", scale=1e3,
          curves=[(GRAV_X, smooth_grav.predicted, BLUE, "this model", "-"),
                  (GRAV_X, flat.predicted, ORANGE, "no depth weighting", "--")],
          title="Both fit equally well")
fig.tight_layout()

The anomaly is now centred on the tunnel, at the right depth — and it fits
the data no better and no worse than the shallow model did. Depth
weighting does not come from the data; it is a choice that cancels a known
bias. It is a good choice, but a choice.

Notice also what the model still gets wrong. The real tunnel is a sharp
box with Δρ = −2.65; the recovered one is a soft blob several metres
across with a contrast **ten times weaker**. That is the smoothness we
asked for. We will come back to it in section 5.

### 4.3 Choosing β

So far β has been set so that $\phi_d = N$: the **discrepancy principle**.
The cell below runs the inversion over a wide range of β and plots the
trade-off. Read the left panel from left to right: as β grows the model
gets simpler ($\phi_m$ falls) and the fit gets worse ($\phi_d$ rises). The
**Tikhonov curve** on the right is the same thing drawn as $\phi_m$ against
$\phi_d$; its corner is another common way to choose β.

The models underneath show what happens either side (each has its own
colour scale). With β too small the inversion fits the noise and invents
structure to do it — the blobs near the surface are noise, not geology.
With β too large it is too smooth and too weak to fit the anomaly at all.

In [ ]:
sweep = inv_grav.sweep()
beta_star = smooth_grav.beta

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].loglog(sweep["beta"], sweep["phi_d"], color=BLUE, lw=2, label="φ_d, data misfit")
axs[0].loglog(sweep["beta"], sweep["phi_m"] * sweep["phi_d"].max() / sweep["phi_m"].max(),
              color=ORANGE, lw=2, ls="--", label="φ_m, model objective (scaled)")
axs[0].axhline(GRAV_X.size, color="0.5", lw=0.8)
axs[0].text(sweep["beta"].min(), GRAV_X.size * 1.2, "target φ_d = N", color="0.35", fontsize=9)
axs[0].axvline(beta_star, color="0.5", lw=0.8, ls=":")
axs[0].set(xlabel="β", ylabel="objective", title="Fit against simplicity")
axs[0].legend(frameon=False, fontsize=9, loc="lower right")

axs[1].loglog(sweep["phi_d"], sweep["phi_m"], color=BLUE, lw=2)
axs[1].plot(smooth_grav.phi_d, smooth_grav.phi_m, "o", color=INK, ms=7)
axs[1].annotate("φ_d = N", (smooth_grav.phi_d, smooth_grav.phi_m), xytext=(8, 6),
                textcoords="offset points")
axs[1].set(xlabel="φ_d", ylabel="φ_m", title="Tikhonov curve")
fig.tight_layout()

fig, axs = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, factor, name in zip(axs, [1 / 100, 1, 100], ["β / 100: fits the noise",
                                                    "β at φ_d = N",
                                                    "β × 100: too smooth"]):
    r = inv_grav.solve(beta_star * factor)
    section(ax, r.model, title=f"{name}   (φ_d/N = {r.chi2:.2f})", label="Δρ (g/cm³)")
fig.tight_layout()

### 4.4 The magnetic inversion

The same recipe for the magnetic data: depth weighting with $\nu = 1$, β
from the discrepancy principle.

Underneath each model are the **normalised residuals**,
$(d^{\text{pred}}_i - d^{\text{obs}}_i)/\sigma_i$. If the model explains
everything but the noise they scatter between about −2 and +2 with no
pattern; a run of residuals of one sign means the model is missing
something real.

In [ ]:
inv_mag = LinearInversion(mesh, G_mag, mag_obs, MAG_STD, weighting="magnetics", z0=MAG_HEIGHT)
smooth_mag = inv_mag.solve()
print(f"beta = {smooth_mag.beta:.3g}, phi_d / N = {smooth_mag.chi2:.2f}")


def residuals(ax, x, result, obs, std, title):
    r = (result.predicted - obs) / std
    ax.axhspan(-2, 2, color="0.93", zorder=0)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.plot(x, r, "o", ms=3.5, color=INK)
    ax.set(ylim=(-4, 4), xlabel="distance along profile (m)",
           ylabel="(pred − obs) / σ", title=title)


fig, axs = plt.subplots(2, 2, figsize=(13, 6.4), gridspec_kw={"height_ratios": [1.3, 1]})
section(axs[0, 0], smooth_grav.model, title="Density contrast (smooth)", label="Δρ (g/cm³)")
section(axs[0, 1], smooth_mag.model, title="Susceptibility contrast (smooth)", label="Δχ (SI)")
residuals(axs[1, 0], GRAV_X, smooth_grav, grav_obs, GRAV_STD, "Gravity residuals")
residuals(axs[1, 1], MAG_X, smooth_mag, mag_obs, MAG_STD, "Magnetic residuals")
fig.tight_layout()

Both methods put the tunnel in the right place. The residuals are mostly
noise, but not entirely: look at the gravity residuals over the tunnel,
where three in a row are positive. The predicted anomaly is not as deep as
the observed one, because a smooth model cannot make an anomaly as sharp as
a compact body does. The discrepancy principle only controls the *total*
misfit, so it allows this; a run of same-sign residuals is the thing to
look for. The magnetic model also has weak positive side-lobes, which the
real ground does not.

Both are also badly **blurred and too weak**. The smoothness term is the
reason, and the next section changes it.

## 5. Adding what we know

A smooth model is the *most cautious* answer — it adds no structure the data
do not insist on. But we know more than the inversion does, and we can tell
it.

### 5.1 Bounds

We are looking for a void in rock. Wherever there is a contrast, it can
only be **negative** (nothing in the ground is denser or more magnetic than
the rock around it) and no more negative than **empty space**:

$$-2.65 \le \Delta\rho \le 0, \qquad -0.005 \le \Delta\chi \le 0.$$

With bounds there is no longer a closed-form answer; the inversion uses
L-BFGS-B, a quasi-Newton method that respects bounds, and searches for the
β at which $\phi_d = N$ again. This takes a few seconds per method.

In [ ]:
rho_bounds = (AIR[0] - ROCK[0], 0.0)
chi_bounds = (AIR[1] - ROCK[1], 0.0)
bounded_grav = inv_grav.solve(bounds=rho_bounds)
bounded_mag = inv_mag.solve(bounds=chi_bounds)
for name, r in [("gravity", bounded_grav), ("magnetics", bounded_mag)]:
    print(f"{name:9s}: phi_d / N = {r.chi2:.2f}, model from {r.model.min():.3g} to {r.model.max():.3g}")

fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
section(axs[0], bounded_grav.model, vmax=abs(rho_bounds[0]) / 4,
        title="Density contrast (smooth, bounded)", label="Δρ (g/cm³)")
section(axs[1], bounded_mag.model, vmax=abs(chi_bounds[0]) / 4,
        title="Susceptibility contrast (smooth, bounded)", label="Δχ (SI)")
fig.tight_layout()

The bounds remove the positive side-lobes and concentrate the anomaly a
little, but the image is still a smooth blob: the smoothness term is still
in charge.

### 5.2 Compactness

A tunnel is not a smooth gradient; it is a compact body with sharp edges.
**Minimum-support** regularisation (Last and Kubik, 1983; Portniaguine and
Zhdanov, 1999) asks for the model with the **fewest cells away from zero**
instead of the smallest values. It replaces the smallness term with

$$\int \frac{(w\,m)^2}{m^2 + \varepsilon^2}\,dA,$$

which charges about the same for any value much larger than ε. That makes
it cheap to concentrate the anomaly in a few strong cells and expensive to
smear it over many weak ones. Because $m$ appears in the weight, it is
solved by **iterative reweighting**: start from the smooth model, re-weight
every cell by $\varepsilon^2/(m^2+\varepsilon^2)$, solve again, repeat. A
little smoothness is kept so the result is not a scatter of single cells.

ε is the size below which a value counts as "zero". Here it is 5 % of the
contrast we expect for a void. The table shows the iterations: the strongest
value grows each time, while β is adjusted to keep $\phi_d \approx N$.

In [ ]:
compact_grav = inv_grav.solve_compact(eps=0.05 * abs(rho_bounds[0]), bounds=rho_bounds)
compact_mag = inv_mag.solve_compact(eps=0.05 * abs(chi_bounds[0]), bounds=chi_bounds)
display(compact_grav.info["history"].round(3).set_index("iteration").T)

fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
section(axs[0], compact_grav.model, vmax=abs(rho_bounds[0]),
        title=f"Density contrast (compact)   φ_d/N = {compact_grav.chi2:.2f}", label="Δρ (g/cm³)")
section(axs[1], compact_mag.model, vmax=abs(chi_bounds[0]),
        title=f"Susceptibility contrast (compact)   φ_d/N = {compact_mag.chi2:.2f}", label="Δχ (SI)")
fig.tight_layout()

### 5.3 All the models side by side

Every model below fits the data to within the noise. They are drawn on the
**same colour scale** as the true model, so the difference in strength is
visible directly.

In [ ]:
models = {
    "true": (true_rho, true_chi),
    "smooth": (smooth_grav.model, smooth_mag.model),
    "smooth + bounds": (bounded_grav.model, bounded_mag.model),
    "compact + bounds": (compact_grav.model, compact_mag.model),
}
fig, axs = plt.subplots(len(models), 2, figsize=(13, 2.9 * len(models)))
for row, (name, (rho, chi)) in enumerate(models.items()):
    section(axs[row, 0], rho, vmax=abs(rho_bounds[0]), title=f"Δρ — {name}", label="g/cm³")
    section(axs[row, 1], chi, vmax=abs(chi_bounds[0]), title=f"Δχ — {name}", label="SI")
fig.tight_layout()

### 5.4 What the data actually determine

Some properties of the recovered models change a lot from one set of rules
to the next; others hardly move. The table measures a few of them for every
model, within 15 m of the tunnel:

* **Centre x and depth** — where the anomaly is: the centre of the
  negative part of the model, weighted by contrast squared.
* **Peak contrast** — how strong the anomaly is, as a fraction of the truth.
* **Apparent size** — the area of the cells stronger than half the peak.
  The true tunnel is 12 m².
* **Total contrast × area** — for gravity, the missing mass per metre of
  tunnel (in g/cm³ · m², i.e. tonnes per metre). By Gauss's theorem the
  total anomalous mass is fixed by the data, whatever its shape, so this is
  the number gravity should get right.

In [ ]:
near = mesh.core(TUNNEL_X - 15, TUNNEL_X + 15, 15)
xc, zc_cells = mesh.centers()
rows = []
for name, (rho, chi) in models.items():
    for method, m, truth in [("gravity", rho, true_rho), ("magnetics", chi, true_chi)]:
        wgt = np.where(near, np.clip(m, None, 0), 0) ** 2 * mesh.area
        peak = m[near].min()
        rows.append({
            "model": name, "method": method,
            "centre x (m)": np.sum(wgt * xc) / wgt.sum(),
            "centre depth (m)": np.sum(wgt * zc_cells) / wgt.sum(),
            "peak / true peak": peak / truth.min(),
            "apparent size (m²)": mesh.area[near & (m <= peak / 2)].sum(),
            "total / true total": np.sum((m * mesh.area)[near]) / np.sum(truth * mesh.area),
        })
summary = pd.DataFrame(rows).set_index(["method", "model"]).sort_index(level=0, sort_remaining=False)
print(f"true tunnel: centre x = {TUNNEL_X:g} m, depth {tunnel_top:g}-{tunnel_bottom:g} m; "
      f"missing mass {-np.sum(true_rho * mesh.area):.1f} t per metre of tunnel")
display(summary.round(2))

* **Horizontal position is robust.** Every model is centred within half a
  metre of the tunnel.
* **Depth is less so.** The smooth models put the centre 1–4 m too deep:
  they smear the anomaly downward, where depth weighting makes values
  cheap. The compact models are within about a metre.
* **Peak contrast is not.** It ranges from about a tenth of the truth to
  most of it, depending only on the rules. The smooth model is not
  "less accurate" data processing — it is a different, equally valid answer
  to a question the data cannot settle.
* **Total missing mass is robust for gravity.** The bounded models recover
  it to within roughly ten percent. (The unbounded smooth model is lower:
  it lets a weak positive halo cancel part of the deficit.) The magnetic
  equivalent is much less stable, because a 2-D magnetic survey cannot see a
  laterally uniform change in susceptibility at all.
* **Shape and size are not determined.** The apparent size ranges over
  nearly a factor of ten from one model to the next. The compact models
  come close to 12 m² here, but only because ε was chosen to suit a void;
  their size is set as much by ε as by the data, and the shape is a rounded
  body, not a 4 × 3 m box. Remember the line-mass comparison in section 2.

## 6. The density and susceptibility models

The inversion recovers **contrasts**, because that is all the data see. To
turn them into a density model and a susceptibility model of the ground,
add them back to the layered background: soil over rock. The background
has to come from somewhere else — the soil thickness from refraction
seismics or an auger hole, the rock properties from samples or tables. It
is *not* something these data can supply (section 1).

The final figure puts the compact, bounded models on that background, with
their fit to the data underneath.

In [ ]:
final_density = background_rho + compact_grav.model
final_susceptibility = background_chi + compact_mag.model

fig, axs = plt.subplots(2, 2, figsize=(13, 6.6), gridspec_kw={"height_ratios": [1.15, 1]})
section(axs[0, 0], final_density, cmap="Blues", vmin=0, vmax=2.8,
        title="Density from gravity", label="g/cm³")
section(axs[0, 1], final_susceptibility, cmap="Purples", vmin=0, vmax=0.0055,
        title="Susceptibility from magnetics", label="SI")
plot_data(axs[1, 0], GRAV_X, grav_obs, GRAV_STD, unit="gravity anomaly (µGal)", scale=1e3,
          curves=[(GRAV_X, compact_grav.predicted, BLUE, "predicted", "-")],
          title=f"Gravity fit, φ_d/N = {compact_grav.chi2:.2f}")
plot_data(axs[1, 1], MAG_X, mag_obs, MAG_STD, unit="total-field anomaly (nT)",
          curves=[(MAG_X, compact_mag.predicted, BLUE, "predicted", "-")],
          title=f"Magnetic fit, φ_d/N = {compact_mag.chi2:.2f}")
for ax in axs[1]:
    ax.set_xlim(-20, 20)
fig.tight_layout()

Two independent methods, measuring two different physical properties, put
a low-density, low-susceptibility body in the same place. That agreement
is worth more than either image alone. The natural next step is to invert
the two data sets **together**, asking that the density and susceptibility
models share the same structure (a *cross-gradient* constraint) — that is
the joint modelling this package is building toward.

## 7. Things to try

Change a number in section 1 or 3 and use **Run > Run All Cells** (or
**Runtime > Run all** in Colab). Each takes under a minute.

1. **Ozark bedrock.** Much of Missouri is Ordovician dolomite, with a
   susceptibility nearer 0.0001 SI. Set `ROCK = (2.80, 0.0001)`. How big is
   the magnetic anomaly now, compared with the 0.5 nT noise? What does the
   magnetic inversion return? (Would you have known it was empty without
   the true model to compare against?)
2. **A deeper tunnel.** Set `TUNNEL_ROOF_BELOW_ROCK = 8`. Which method
   loses the tunnel first, and why? (Look at the sensitivity plot again.)
3. **Sparser stations.** Gravity every 5 m: `GRAV_X = np.arange(-30, 30.01, 5.0)`.
   Use the half-width rule to work out the largest station spacing that
   still resolves the anomaly.
4. **Sensor height.** Put the magnetometer on a full 2 m staff. The
   anomaly gets smaller but so does the noise from magnetic stones in the
   soil — real surveyors make this trade.
5. **Profile direction.** Run the profile west to east,
   `PROFILE_AZIMUTH = 90.0`. The tunnel now runs north–south. What
   happens to the magnetic anomaly, and is the magnetic inversion better or
   worse?
6. **Getting the noise wrong.** Leave the noise as it is but tell the
   inversion it is half as large (pass `GRAV_STD / 2` to `LinearInversion`).
   It will now fit the data more closely than the noise allows. What
   appears in the model?
7. **ε.** In section 5.2, try `eps` at 1 % and 20 % of the contrast. Which
   features of the compact model change, and which stay?
8. **A second tunnel**, 10 m to the north: add another `mesh.rectangle(...)`
   to `tunnel` in section 1. How close can two tunnels be before the
   anomalies merge?

### References

* Blakely, R. J. (1995). *Potential Theory in Gravity and Magnetic
  Applications*. Cambridge University Press.
* Last, B. J., and Kubik, K. (1983). Compact gravity inversion.
  *Geophysics* 48, 713–721.
* Li, Y., and Oldenburg, D. W. (1996). 3-D inversion of magnetic data.
  *Geophysics* 61, 394–408.
* Li, Y., and Oldenburg, D. W. (1998). 3-D inversion of gravity data.
  *Geophysics* 63, 109–119.
* Portniaguine, O., and Zhdanov, M. S. (1999). Focusing geophysical
  inversion images. *Geophysics* 64, 874–887.
* Telford, W. M., Geldart, L. P., and Sheriff, R. E. (1990). *Applied
  Geophysics*, 2nd ed. Cambridge University Press.